# exp_20260516_032_custom_realmlp_cv095352_min

Source: `pss6e5-realmlp-cv-0-95352.ipynb`.

Custom Torch RealMLP branch. This is not pytabkit RealMLP_TD.

Role: RealMLP-branch diversity / replacement candidate. Judge mainly by corr vs 029 and blend weight.


In [1]:
# ============================================================
# PS S6E5 - exp_20260516_032_custom_realmlp_cv095352_min
#
# Source:
#   pss6e5-realmlp-cv-0-95352.ipynb
#
# Purpose:
#   Run custom Torch RealMLP as a RealMLP-branch diversity / replacement candidate.
#
# Notes:
#   - This is NOT pytabkit RealMLP_TD. It is a hand-written Torch RealMLP-like model.
#   - Fold split follows the source code: index % 5, not StratifiedKFold.
#   - PitStop is used if present in competition train/test.
#   - Normalized_TyreLife from original is not used because FEATURES are derived from test columns.
#   - OOF/pred .npy, submission, cv_result.json, and memo_draft.yml are saved.
# ============================================================

In [2]:
import os
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260516_032_custom_realmlp_cv095352_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_PATH = Path(
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
    )

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    SEED = 2026

    num_folds = 5
    epochs = 15
    train_bs = 1024
    eval_bs = 10240
    embed_dim = 4
    LR = 0.003
    n_ens = 32
    onehotmax = 10
    ls = 0.03

    # Source code uses index % fold splitting.
    fold_split = "index_mod_5"

    # Slight safety/correctness improvement:
    # source saved best checkpoint but assigned OOF from last evaluated epoch.
    # Here we reload best checkpoint and recompute validation prediction for OOF/test.
    use_best_checkpoint_for_oof = True


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)


# expose source-like variable names
num_folds = CFG.num_folds
epochs = CFG.epochs
train_bs = CFG.train_bs
eval_bs = CFG.eval_bs
embed_dim = CFG.embed_dim
LR = CFG.LR
n_ens = CFG.n_ens
onehotmax = CFG.onehotmax
ls = CFG.ls
target_col = CFG.TARGET

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def per_year_auc(train_df: pd.DataFrame, y_true, pred) -> dict:
    out = {}
    if "Year" not in train_df.columns:
        return out

    y_arr = np.asarray(y_true).astype(int)
    p_arr = np.asarray(pred, dtype=float)

    for yr in sorted(pd.Series(train_df["Year"]).dropna().unique().tolist()):
        mask = train_df["Year"].values == yr
        if mask.sum() == 0 or len(np.unique(y_arr[mask])) < 2:
            out[str(int(yr))] = None
        else:
            out[str(int(yr))] = float(roc_auc_score(y_arr[mask], p_arr[mask]))

    return out


seed_everything(CFG.SEED)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch: 2.10.0+cu128
GPU: Tesla T4


In [5]:
# ============================================================
# Load data and Config
# ============================================================

print_section("Load data")

train_raw = pd.read_csv(CFG.TRAIN_PATH)
test_raw = pd.read_csv(CFG.TEST_PATH)
sample_submission = pd.read_csv(CFG.SUB_PATH)
origin = pd.read_csv(CFG.ORIG_PATH)

train_id = train_raw[CFG.ID].copy()
test_id = test_raw[CFG.ID].copy()

drop_cols = [CFG.ID]
train = train_raw.drop(drop_cols, axis=1)
test = test_raw.drop(drop_cols, axis=1)

print(f"train.shape:{train.shape}, test.shape:{test.shape}, origin.shape:{origin.shape}")
print("train target rate:", train[target_col].mean())
print("origin target rate:", origin[target_col].mean())


Load data
train.shape:(439140, 15), test.shape:(188165, 14), origin.shape:(101371, 16)
train target rate: 0.19898210137996994
origin target rate: 0.25479673673930414


In [6]:
# ============================================================
# Feature Engineering
# ============================================================

print_section("Feature Engineering")

NUMS = [c for c in test.columns if test[c].dtype != object]
M = train[NUMS].max()

def x2(x):
    if x <= 50:
        y = -0.000001 * x**2 - 0.001191 * x + 0.195376
    else:
        y = -0.000001 * x**2 - 0.001288 * x + 0.184553
    return (y + 4.4) ** 2

def FE(df):
    df = df.copy()

    df["Race_Year"] = df["Race"].astype(str) + df["Year"].astype(str)
    df["Race_Compound"] = df["Race"].astype(str) + df["Compound"].astype(str)
    df["_LapNumber_/_RaceProgress"] = (
        df["LapNumber"] / (df["RaceProgress"] + 1e-6)
    ).astype("float32")
    df["_TyreLife_/_LapNumber"] = (
        df["TyreLife"] / df["LapNumber"].clip(lower=1)
    ).astype("float32")
    df["_LapNumber_/_RaceProgress_cat"] = df["_LapNumber_/_RaceProgress"].round(2).astype(str)
    df["_TyreLife_/_LapNumber_cat"] = df["_TyreLife_/_LapNumber"].round(2).astype(str)

    df["_LapTime (s)_*_Cumulative_Degradation"] = (
        df["LapTime (s)"] * df["Cumulative_Degradation"]
    ).astype("float32")
    df["_LapTime (s)_*_Cumulative_Degradation_abs"] = (
        df["LapTime (s)"] * df["Cumulative_Degradation"].abs()
    ).astype("float32")
    df["_LapTime (s)_/_Cumulative_Degradation_abs"] = (
        df["LapTime (s)"] / (df["Cumulative_Degradation"].abs() + 1e-6)
    ).astype("float32")

    for c in NUMS:
        if c not in df.columns:
            continue
        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

    df["LapTime_Delta"] = df["LapTime_Delta"].apply(lambda x: x2(x))
    df["driver0"] = df["Driver"].apply(lambda x: x[0])
    df["LapNumber"] = (df["LapNumber"] - 50) ** 2
    df["RaceProgress"] = (df["RaceProgress"] - 0.6) ** 2

    df["LapTime (s)_cat"] = df["LapTime (s)"].astype(str)
    df["LapTime_Delta_cat"] = (df["LapTime_Delta"] // 2).astype(int).astype(str)
    df["Cumulative_Degradation_cat"] = (df["Cumulative_Degradation"] // 2).astype(int).astype(str)

    for c in ["Year", "LapNumber"]:
        df[c + "_cat"] = df[c].astype(str)

    return df

train = FE(train)
test = FE(test)
origin = FE(origin)

CATS = [c for c in test.columns if test[c].dtype == object or train[c].nunique() <= onehotmax]
NUMS = [c for c in test.columns if c not in CATS]

print("n CATS:", len(CATS))
print("n NUMS:", len(NUMS))
print("CATS:", CATS[:20])
print("NUMS:", NUMS[:20])


Feature Engineering
n CATS: 16
n NUMS: 13
CATS: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'Stint', 'Race_Year', 'Race_Compound', '_LapNumber_/_RaceProgress_cat', '_TyreLife_/_LapNumber_cat', 'driver0', 'LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'Year_cat', 'LapNumber_cat']
NUMS: ['LapNumber', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs']


In [7]:
# ============================================================
# LabelEncoder-like mapping from train
# ============================================================

print_section("Label Encoding")

for c in CATS:
    U = train[c].unique()
    mapping = {v: i + 1 for i, v in enumerate(U)}
    train[c] = train[c].map(mapping)
    test[c] = test[c].map(mapping).fillna(0)
    origin[c] = origin[c].map(mapping).fillna(0)

FEATURES = CATS + NUMS

# Guardrail: ensure Normalized_TyreLife is not in FEATURES.
assert CFG.DANGER_COL not in FEATURES, f"{CFG.DANGER_COL} must not be used"
assert CFG.DANGER_COL not in test.columns, f"{CFG.DANGER_COL} must not be in test-derived features"

pd.Series(FEATURES).to_csv(CFG.OUTDIR / "feature_columns.csv", index=False, header=False)


Label Encoding


In [8]:
# ============================================================
# Data preprocessing
# ============================================================

class RobustScaleSmoothClipTransform(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        assert isinstance(X, np.ndarray)

        self._median = np.median(X, axis=-2)
        quant_diff = np.quantile(X, 0.75, axis=-2) - np.quantile(X, 0.25, axis=-2)

        idxs = quant_diff == 0.0
        quant_diff[idxs] = 0.5 * (np.max(X, axis=-2)[idxs] - np.min(X, axis=-2)[idxs])

        factors = 1.0 / (quant_diff + 1e-30)
        factors[quant_diff == 0.0] = 0.0
        self._factors = factors
        return self

    def transform(self, X, y=None):
        x_scaled = self._factors[None, :] * (X - self._median[None, :])
        return x_scaled / np.sqrt(1 + (x_scaled / 3) ** 2)

In [9]:
# ============================================================
# Model
# ============================================================

class ScalingLayer(nn.Module):
    def __init__(self, n_ens: int, n_features: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.scale[None, :, :]


class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens: int, cat_dims, embed_dim=8, device=None):
        super().__init__()
        self.n_ens = n_ens
        self.device = device
        self.cat_dims = cat_dims

        self.onehot_features = []
        self.embed_features = []
        self.embed_dims = []
        self.embed_offsets = []

        for i, dim in enumerate(cat_dims):
            if dim <= onehotmax:
                self.onehot_features.append(i)
            else:
                self.embed_features.append(i)
                self.embed_dims.append(dim)

        if self.embed_features:
            total_vocab = sum(self.embed_dims) * n_ens
            self.combined_emb = nn.Embedding(total_vocab, embed_dim, padding_idx=0)

            offset = 0
            for dim in self.embed_dims:
                self.embed_offsets.append(offset)
                offset += dim
            self.per_ens_offset = sum(self.embed_dims)

    def forward(self, x):
        batch_size, n_ens, n_cat = x.shape
        features = []

        if self.onehot_features:
            onehot_x = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_onehot_dim = sum(onehot_dims)

            onehot_encoded = torch.zeros(batch_size, n_ens, total_onehot_dim, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx : idx + 1].long()
                onehot_encoded.scatter_(2, pos + start, 1.0)
                start += dim

            features.append(onehot_encoded)

        if self.embed_features:
            batch_size, n_ens, n_cat = x.shape
            n_embed_feat = len(self.embed_features)

            embed_x = x[:, :, self.embed_features].long()

            ens_offset = torch.arange(n_ens, device=x.device) * self.per_ens_offset
            feat_offset = torch.tensor(self.embed_offsets, device=x.device)

            indices = embed_x + feat_offset.unsqueeze(0).unsqueeze(0) + ens_offset.unsqueeze(0).unsqueeze(-1)

            embedded = self.combined_emb(indices)
            embedded = embedded.view(batch_size, n_ens, -1)
            features.append(embedded)

        return torch.cat(features, dim=2)


class PBLDEmbedding(nn.Module):
    def __init__(
        self,
        n_ens: int,
        n_features: int,
        hidden_dim: int = 16,
        out_dim: int = 4,
        freq_scale: float = 0.1,
    ):
        super().__init__()
        self.n_ens = n_ens
        self.n_features = n_features
        self.out_dim = out_dim
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) * (1.0 / np.sqrt(hidden_dim)))
        self.b2 = nn.Parameter(torch.randn(n_ens, n_features, out_dim - 1))

        self.act = nn.GELU()
        nn.init.uniform_(self.b1, -np.pi, np.pi)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        x_expanded = x.unsqueeze(-1)
        w1_expanded = self.w1.unsqueeze(0)
        b1_expanded = self.b1.unsqueeze(0)
        periodic = torch.cos(2 * np.pi * (x_expanded * w1_expanded + b1_expanded))
        transformed = torch.einsum("b n f h, n f h d -> b n f d", periodic, self.w2)
        transformed = self.act(transformed + self.b2.unsqueeze(0))
        result = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        return result.view(batch_size, self.n_ens, -1)


class NTPLinear(nn.Module):
    def __init__(self, n_ens: int, in_features: int, out_features: int, bias: bool = True) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.ndim == 3
        x = torch.einsum("b n i, n i o -> b n o", x, self.weight) / np.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class RealMLP(torch.nn.Module):
    def __init__(self, output_dim=2, cat_dims=[], n_numerical=None, n_ens=8, embed_dim=4):
        super().__init__()
        act = nn.GELU
        self.n_ens = n_ens
        self.embed_dim = embed_dim

        self.cate = CategoricalFeatureLayer(
            n_ens=self.n_ens,
            cat_dims=cat_dims,
            embed_dim=self.embed_dim,
            device=device,
        )

        self.num_embed = PBLDEmbedding(
            n_features=n_numerical,
            hidden_dim=16,
            out_dim=4,
            freq_scale=1.0,
            n_ens=self.n_ens,
        )

        num_emb_dim = n_numerical * 4
        cat_emb_dim = sum([c if c <= onehotmax else self.embed_dim for c in cat_dims])
        total_dim = num_emb_dim + cat_emb_dim

        self.dropout = nn.Dropout(0.02)
        self.model = nn.Sequential(
            ScalingLayer(n_ens=self.n_ens, n_features=total_dim),
            NTPLinear(n_ens=self.n_ens, in_features=total_dim, out_features=256),
            act(),
            self.dropout,
            NTPLinear(n_ens=self.n_ens, in_features=256, out_features=128),
            act(),
            self.dropout,
        )
        self.output = NTPLinear(n_ens=self.n_ens, in_features=128, out_features=output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)

        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)

        combined = torch.cat([x_num, x_cat], dim=2)

        x = self.model(combined)
        x = self.output(x)
        return F.softmax(x, dim=2)

In [10]:
# ============================================================
# Training helpers
# ============================================================

def get_parameter_groups_3class(model):
    scale_params = []
    weight_params = []
    bias_params = []

    for name, param in model.named_parameters():
        if "scale" in name:
            scale_params.append(param)
        elif "bias" in name:
            bias_params.append(param)
        else:
            weight_params.append(param)

    return scale_params, weight_params, bias_params


def flat_anneal(init_value, progress, flat_ratio=0.5):
    if progress < flat_ratio:
        return init_value
    else:
        decay_progress = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - decay_progress)


def cos(init_value, progress):
    return init_value * (math.cos(math.pi * progress) + 1) / 2


def cross_entropy_loss(y_true, y_pred, label_smoothing=0.04, eps=1e-15):
    p = y_pred[:, 1]
    p = torch.clip(p, eps, 1 - eps)

    noise = torch.randn_like(y_true.float()) * 0.01
    smooth = label_smoothing + noise
    smooth = torch.clip(smooth, 0.00001, 0.1)

    y_true_smooth = y_true * (1 - smooth) + smooth / 2
    ce_loss = -(y_true_smooth * torch.log(p) + (1 - y_true_smooth) * torch.log(1 - p))

    return torch.mean(ce_loss)


def predict_in_batches(model, X_num_t, X_cat_t, eval_bs, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for idx in range(0, len(X_num_t), eval_bs):
            batch_x_num = X_num_t[idx : idx + eval_bs]
            batch_x_cat = X_cat_t[idx : idx + eval_bs]
            probs = model(batch_x_num, batch_x_cat).mean(dim=1)

            if not torch.isfinite(probs).all():
                n_bad = (~torch.isfinite(probs)).sum().item()
                raise RuntimeError(f"Non-finite prediction detected: {n_bad} values")

            preds.append(probs.cpu().numpy())

    return np.concatenate(preds, axis=0)

In [11]:
# ============================================================
# CV Training
# ============================================================

print_section("CV Training")

X_num = train[NUMS].values
X_cat = train[CATS].values
y = train[target_col].values
full_idx = np.arange(len(y))

X_num_origin = origin[NUMS].values
X_cat_origin = origin[CATS].values
y_origin = origin[target_col].values

testX_num = test[NUMS].values
testX_cat = test[CATS].values

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device:{device}")

oof = np.zeros(len(train), dtype=np.float32)
test_preds_folds = []
fold_scores = []
fold_best_epochs = []

for fold in range(num_folds):
    print(f"Fold {fold} -------------------------------------->")

    valid_idx = np.array([index for index in full_idx if index % num_folds == fold])
    train_idx_fold = np.array([index for index in full_idx if index % num_folds != fold])

    X_num_train, X_cat_train, y_train = X_num[train_idx_fold], X_cat[train_idx_fold], y[train_idx_fold]
    X_num_val, X_cat_val, y_val = X_num[valid_idx], X_cat[valid_idx], y[valid_idx]

    X_num_train = np.concatenate((X_num_train, X_num_origin), axis=0)
    X_cat_train = np.concatenate((X_cat_train, X_cat_origin), axis=0)
    y_train = np.concatenate((y_train, y_origin), axis=0)

    cols_to_encode = [i for i in range(X_cat_train.shape[1]) if X_cat_train[:, i].max() > onehotmax]

    if len(cols_to_encode) > 0:
        te = TargetEncoder(target_type="binary", cv=5)
        X_cat_train_encoded = te.fit_transform(X_cat_train[:, cols_to_encode], y_train)
        X_cat_val_encoded = te.transform(X_cat_val[:, cols_to_encode])
        X_cat_test_encoded = te.transform(testX_cat[:, cols_to_encode])

        X_num_train = np.concatenate([X_num_train, X_cat_train_encoded], axis=1)
        X_num_val = np.concatenate([X_num_val, X_cat_val_encoded], axis=1)
        X_num_test = np.concatenate([testX_num, X_cat_test_encoded], axis=1)
    else:
        X_num_test = testX_num.copy()

    print("NaN check before torch tensors")
    print("X_num_train nan:", np.isnan(X_num_train).sum())
    print("X_num_val nan:", np.isnan(X_num_val).sum())
    print("X_num_test nan:", np.isnan(X_num_test).sum())
    print("X_num_train inf:", np.isinf(X_num_train).sum())
    print("X_num_val inf:", np.isinf(X_num_val).sum())
    print("X_num_test inf:", np.isinf(X_num_test).sum())

    rssc = RobustScaleSmoothClipTransform()
    rssc.fit(X_num_train)
    
    X_num_train = rssc.transform(X_num_train)
    X_num_val = rssc.transform(X_num_val)
    X_num_test = rssc.transform(X_num_test)
    
    print("After RSSC")
    print("X_num_train nan:", np.isnan(X_num_train).sum(), "inf:", np.isinf(X_num_train).sum())
    print("X_num_val nan:", np.isnan(X_num_val).sum(), "inf:", np.isinf(X_num_val).sum())
    print("X_num_test nan:", np.isnan(X_num_test).sum(), "inf:", np.isinf(X_num_test).sum())

    train_idx_local = np.arange(len(y_train))

    realmlp = RealMLP(
        output_dim=2,
        cat_dims=X_cat.max(axis=0) + 1,
        n_numerical=X_num_train.shape[1],
        n_ens=n_ens,
        embed_dim=embed_dim,
    ).to(device)

    best_auc = 0.0
    best_epoch = None
    checkpoint_path = CFG.OUTDIR / f"RealMLP_fold{fold}.pth"

    scale_params, weights, biases = get_parameter_groups_3class(realmlp)

    optimizer = torch.optim.AdamW(
        [
            {"params": scale_params, "name": "scale", "lr": LR * 10.0, "weight_decay": 1e-2 * 0.1},
            {"params": weights, "name": "weights", "lr": LR, "weight_decay": 1e-2},
            {"params": biases, "name": "biases", "lr": LR * 0.1, "weight_decay": 1e-2 * 0.5},
        ],
        betas=(0.9, 0.98),
    )

    X_num_train_t = torch.as_tensor(X_num_train, dtype=torch.float32, device=device)
    X_cat_train_t = torch.as_tensor(X_cat_train, dtype=torch.long, device=device)
    y_train_t = torch.as_tensor(y_train, dtype=torch.long, device=device)

    X_num_val_t = torch.as_tensor(X_num_val, dtype=torch.float32, device=device)
    X_cat_val_t = torch.as_tensor(X_cat_val, dtype=torch.long, device=device)

    X_num_test_t = torch.as_tensor(X_num_test, dtype=torch.float32, device=device)
    X_cat_test_t = torch.as_tensor(testX_cat, dtype=torch.long, device=device)

    for epoch in range(epochs):
        for idx in range(0, len(y_train_t), train_bs):
            progress = epoch / epochs + idx / (len(y_train_t) * epochs)
            bs_idx = train_idx_local[idx : idx + train_bs]

            realmlp.train()
            optimizer.param_groups[0]["lr"] = flat_anneal(LR * 10.0, progress)
            optimizer.param_groups[1]["lr"] = flat_anneal(LR, progress)
            optimizer.param_groups[2]["lr"] = flat_anneal(LR * 0.1, progress)

            optimizer.zero_grad()

            batch_x_num = X_num_train_t[bs_idx]
            batch_x_cat = X_cat_train_t[bs_idx]
            batch_y = y_train_t[bs_idx]

            y_pred = realmlp(batch_x_num, batch_x_cat)
            loss = cross_entropy_loss(
                batch_y.repeat_interleave(n_ens),
                y_pred.reshape(-1, 2),
                label_smoothing=cos(ls, progress),
            )

            if not torch.isfinite(loss):
                print(f"WARNING: non-finite loss skipped | fold={fold}, epoch={epoch}, idx={idx}")
                optimizer.zero_grad(set_to_none=True)
                continue
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(realmlp.parameters(), max_norm=1.0)
            optimizer.step()

        # source only evaluates in the final 3 epochs
        if epoch >= epochs - 3:
            val_probs = predict_in_batches(realmlp, X_num_val_t, X_cat_val_t, eval_bs, device)
            auc = roc_auc_score(y_val, val_probs[:, 1])

            if auc > best_auc:
                best_auc = auc
                best_epoch = epoch
                torch.save(realmlp.state_dict(), checkpoint_path)
                print(f"fold:{fold}, epoch:{epoch}, best_auc:{best_auc:.6f}")

        np.random.shuffle(train_idx_local)

    # Use best checkpoint for OOF and test prediction.
    if CFG.use_best_checkpoint_for_oof and checkpoint_path.exists():
        realmlp.load_state_dict(torch.load(checkpoint_path, map_location=device))

    val_probs = predict_in_batches(realmlp, X_num_val_t, X_cat_val_t, eval_bs, device)
    fold_auc = roc_auc_score(y_val, val_probs[:, 1])

    oof[valid_idx] = val_probs[:, 1].astype(np.float32)

    test_probs = predict_in_batches(realmlp, X_num_test_t, X_cat_test_t, eval_bs, device)
    test_preds_folds.append(test_probs[:, 1].astype(np.float32))

    fold_scores.append(float(fold_auc))
    fold_best_epochs.append(None if best_epoch is None else int(best_epoch))

    print(f"Fold {fold} final OOF AUC from checkpoint: {fold_auc:.6f}")

    del realmlp
    del X_num_train_t, X_cat_train_t, y_train_t, X_num_val_t, X_cat_val_t, X_num_test_t, X_cat_test_t
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

test_preds = np.mean(test_preds_folds, axis=0).astype(np.float32)

cv_auc = float(roc_auc_score(y, oof))
year_auc = per_year_auc(train_raw, y, oof)

print_section("Result")
print(f"CV:{cv_auc:.9f}")
print("fold_scores:", fold_scores)
print("fold_best_epochs:", fold_best_epochs)
print("per_year_auc:", year_auc)


CV Training
device:cuda
Fold 0 -------------------------------------->
NaN check before torch tensors
X_num_train nan: 0
X_num_val nan: 0
X_num_test nan: 0
X_num_train inf: 0
X_num_val inf: 0
X_num_test inf: 0
After RSSC
X_num_train nan: 0 inf: 0
X_num_val nan: 0 inf: 0
X_num_test nan: 0 inf: 0
fold:0, epoch:12, best_auc:0.953161
fold:0, epoch:13, best_auc:0.953220
fold:0, epoch:14, best_auc:0.953236
Fold 0 final OOF AUC from checkpoint: 0.953236
Fold 1 -------------------------------------->
NaN check before torch tensors
X_num_train nan: 0
X_num_val nan: 0
X_num_test nan: 0
X_num_train inf: 0
X_num_val inf: 0
X_num_test inf: 0
After RSSC
X_num_train nan: 0 inf: 0
X_num_val nan: 0 inf: 0
X_num_test nan: 0 inf: 0
fold:1, epoch:12, best_auc:0.953549
fold:1, epoch:13, best_auc:0.953638
fold:1, epoch:14, best_auc:0.953660
Fold 1 final OOF AUC from checkpoint: 0.953660
Fold 2 -------------------------------------->
NaN check before torch tensors
X_num_train nan: 0
X_num_val nan: 0
X_num_t

In [12]:
# ============================================================
# Save artifacts
# ============================================================

print_section("Save artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", test_preds)

pd.DataFrame({
    CFG.ID: train_id.values,
    target_col: oof,
}).to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub = sample_submission.copy()
target_col_submission = target_col if target_col in sub.columns else [c for c in sub.columns if c != CFG.ID][0]
sub[target_col_submission] = np.clip(test_preds, 1e-7, 1 - 1e-7)

submission_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub.to_csv(submission_path, index=False)

feature_detail = pd.DataFrame({"feature": FEATURES})
feature_detail["is_cat"] = feature_detail["feature"].isin(CATS)
feature_detail["is_num"] = feature_detail["feature"].isin(NUMS)
feature_detail["is_pitstop_related"] = feature_detail["feature"].str.contains("PitStop", regex=False)
feature_detail["is_normalized_tyrelife"] = feature_detail["feature"].eq(CFG.DANGER_COL)
feature_detail.to_csv(CFG.OUTDIR / "feature_columns_detail.csv", index=False)

cv_result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-16",
    },
    "source": {
        "reference_code": "pss6e5-realmlp-cv-0-95352.ipynb",
        "family": "CustomTorchRealMLP",
        "description": "Hand-written Torch RealMLP-like model, not pytabkit RealMLP_TD.",
    },
    "config": {
        "seed": CFG.SEED,
        "num_folds": CFG.num_folds,
        "fold_split": CFG.fold_split,
        "epochs": CFG.epochs,
        "train_bs": CFG.train_bs,
        "eval_bs": CFG.eval_bs,
        "embed_dim": CFG.embed_dim,
        "LR": CFG.LR,
        "n_ens": CFG.n_ens,
        "onehotmax": CFG.onehotmax,
        "ls": CFG.ls,
        "use_best_checkpoint_for_oof": CFG.use_best_checkpoint_for_oof,
    },
    "data": {
        "train_shape": list(train_raw.shape),
        "test_shape": list(test_raw.shape),
        "origin_shape": list(origin.shape),
        "target_rate_train": float(train_raw[target_col].mean()),
        "target_rate_origin": float(origin[target_col].mean()),
    },
    "features": {
        "n_features": len(FEATURES),
        "n_cat": len(CATS),
        "n_num": len(NUMS),
        "danger_col_used": CFG.DANGER_COL in FEATURES,
        "pitstop_used": "PitStop" in FEATURES,
        "cat_features": CATS,
        "num_features": NUMS,
    },
    "result": {
        "cv_auc": cv_auc,
        "public_lb": None,
        "fold_scores": fold_scores,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "fold_best_epochs": fold_best_epochs,
        "per_year_auc": year_auc,
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(submission_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_columns_detail": str(CFG.OUTDIR / "feature_columns_detail.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, ensure_ascii=False, indent=2, default=str)

memo_yml = f"""experiment:
  id: exp_20260516_032_custom_realmlp_cv095352_min
  title: Custom Torch RealMLP CV0.95352
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  status: completed

objective:
  summary: >
    shared_012 custom Torch RealMLPを実行し、029 pytabkit RealMLP clusterとは違う誤差を出せるか確認する。
    単体主力ではなく、RealMLP派生の別実装・相関確認候補として扱う。
  intended_role: realmlp_branch_diversity_candidate

source:
  reference_code: pss6e5-realmlp-cv-0-95352.ipynb
  model_family: CustomTorchRealMLP
  reported_cv_auc: 0.95352
  notes:
    - pytabkit RealMLP_TDではなく、hand-written Torch RealMLP-like model
    - n_ens=32
    - PBLD numerical embedding
    - categorical one-hot/embedding
    - parameter group別AdamW
    - fold split is index % 5, not StratifiedKFold
    - original dataset is appended to each fold train
    - TargetEncoder is fitted inside fold after original append

data:
  competition_train: /kaggle/input/competitions/playground-series-s6e5/train.csv
  competition_test: /kaggle/input/competitions/playground-series-s6e5/test.csv
  original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
  target: PitNextLap
  danger_col:
    name: Normalized_TyreLife
    used: {str(CFG.DANGER_COL in FEATURES).lower()}
    note: >
      FEATURES are derived from test columns, so Normalized_TyreLife from original is not used.

caution:
  fold_split: index_mod_5
  note: >
    CV is not directly comparable to StratifiedKFold-based experiments.
    Judge primarily by correlation and blend weight.
  pitstop_used: {str("PitStop" in FEATURES).lower()}

model:
  family: CustomTorchRealMLP
  seed: {CFG.SEED}
  folds: {CFG.num_folds}
  epochs: {CFG.epochs}
  n_ens: {CFG.n_ens}
  embed_dim: {CFG.embed_dim}
  train_bs: {CFG.train_bs}
  eval_bs: {CFG.eval_bs}
  lr: {CFG.LR}
  label_smoothing_base: {CFG.ls}
  use_best_checkpoint_for_oof: {str(CFG.use_best_checkpoint_for_oof).lower()}

result:
  cv_auc: {cv_auc}
  public_lb: null
  fold_scores: {fold_scores}
  fold_mean: {float(np.mean(fold_scores))}
  fold_std: {float(np.std(fold_scores))}
  fold_best_epochs: {fold_best_epochs}
  per_year_auc: {year_auc}
  compared_to_existing_realmlp:
    "018":
      cv_auc: 0.953475993350039
      public_lb: 0.95316
    "022":
      cv_auc: 0.9534314709948207
      public_lb: 0.95301
    "023":
      cv_auc: 0.9534462611732204
      public_lb: 0.95304
    "027":
      cv_auc: 0.9531400778528742
      public_lb: 0.95260
    "029":
      cv_auc: 0.9540420784720796
      public_lb: 0.95397

artifacts:
  notebook: exp_20260516_032_custom_realmlp_cv095352_min.ipynb
  oof: oof_exp_20260516_032_custom_realmlp_cv095352_min.npy
  pred: pred_exp_20260516_032_custom_realmlp_cv095352_min.npy
  submission: submission_exp_20260516_032_custom_realmlp_cv095352_min.csv
  cv_result: cv_result.json
  feature_columns: feature_columns.csv
  feature_columns_detail: feature_columns_detail.csv

blend_policy:
  benchmark_code: code_010_add_slsqp_drop020.txt
  add_to_stack:
    - "014"
    - "015"
    - "016"
    - "017"
    - "018"
    - "021"
    - "026"
    - "028"
    - "029"
    - "032"
  decision_focus:
    - corr_vs_029
    - corr_vs_018_022_023
    - nm_hc_slsqp_weight
    - logreg_cv
    - public_lb_if_submitted
  keep_rule: >
    Keep if 032 receives meaningful blend weight or provides lower-correlation RealMLP signal.
  drop_rule: >
    Drop if it is highly correlated with 029 and receives near-zero blend weight.

judgement:
  status: pending
  summary: >
    CV/LB/correlation/blend weight確認後に keep / hold / drop を判断する。
"""
with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)

print("Saved to:", CFG.OUTDIR)
print("Submission:", submission_path)
display(sub.head())


Save artifacts
Saved to: /kaggle/working/exp_20260516_032_custom_realmlp_cv095352_min
Submission: /kaggle/working/exp_20260516_032_custom_realmlp_cv095352_min/submission_exp_20260516_032_custom_realmlp_cv095352_min.csv


,id,PitNextLap
0,439140,0.006598
1,439141,0.036088
2,439142,0.007104
3,439143,0.210695
4,439144,0.793072
